In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

interactions = pd.read_csv('../data/cleaned/interactions_clean.csv')
products     = pd.read_csv('../data/cleaned/products_clean.csv')

print(interactions.shape)
interactions.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder

ue = LabelEncoder(); ie = LabelEncoder()
u_idx = ue.fit_transform(interactions['user_id'])
i_idx = ie.fit_transform(interactions['product_id'])

R = csr_matrix((interactions['implicit_score'].values, (u_idx, i_idx)))
density = R.nnz / (R.shape[0] * R.shape[1]) * 100

# Sample 100x100 sub-matrix for visualisation
sample = R[:100, :100].toarray()

plt.figure(figsize=(8, 5))
plt.imshow(sample > 0, cmap='Blues', aspect='auto', interpolation='nearest')
plt.title(f'Interaction matrix sample (100×100)  —  overall density: {density:.2f}%')
plt.xlabel('Product index'); plt.ylabel('User index')
plt.colorbar(label='Has interaction')
plt.tight_layout()
plt.savefig('../data/processed/sparsity_map.png'); plt.show()
print(f'Matrix density: {density:.2f}%')

In [ ]:
raw = pd.read_csv('../data/raw/interactions.csv')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Event type counts
raw['event_type'].value_counts().plot(kind='bar', ax=axes[0], color=['#378ADD','#EF9F27','#1D9E75'])
axes[0].set_title('Events by type'); axes[0].set_xlabel('')

# Implicit score distribution
interactions['implicit_score'].hist(bins=20, ax=axes[1], color='#378ADD', edgecolor='white')
axes[1].set_title('Distribution of implicit scores per (user, product) pair')
axes[1].set_xlabel('Implicit score')

plt.tight_layout()
plt.savefig('../data/processed/event_distribution.png'); plt.show()

In [ ]:
product_pop = (
    interactions.groupby('product_id')['n_events']
    .sum().sort_values(ascending=False).head(15)
    .reset_index()
    .merge(products[['product_id','product_name','category']], on='product_id')
)

plt.figure(figsize=(9, 5))
bars = plt.barh(product_pop['product_name'], product_pop['n_events'], color='#378ADD')
plt.xlabel('Total interactions'); plt.title('Top 15 most interacted products')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../data/processed/top_products.png'); plt.show()
product_pop.head(10)

In [ ]:
cat_counts = products['category'].value_counts()

plt.figure(figsize=(8, 4))
cat_counts.plot(kind='bar', color=sns.color_palette('muted', len(cat_counts)))
plt.title('Products by category'); plt.xlabel('')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../data/processed/category_breakdown.png'); plt.show()

In [ ]:
merged = interactions.merge(products[['product_id','category']], on='product_id')
cat_purchase = merged.groupby('category')['has_purchase'].mean().sort_values(ascending=False)

plt.figure(figsize=(8, 4))
cat_purchase.plot(kind='bar', color='#1D9E75')
plt.title('Purchase conversion rate by category'); plt.ylabel('Avg has_purchase'); plt.xlabel('')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../data/processed/purchase_by_category.png'); plt.show()

print('\nStrategy summary')
print('─' * 40)
print(f"Best purchase category: {cat_purchase.idxmax()} ({cat_purchase.max():.1%})")
print(f"Sparsest category:      {cat_purchase.idxmin()} ({cat_purchase.min():.1%})")